# Personalized Compliment Agent with LangGraph

This notebook demonstrates building a Personalized Compliment Agent using LangGraph.

### Requirements
- **Input**: `{"name": "Bob"}`
- **Output**: `"Bob, you're doing an amazing job learning LangGraph!"`
- **Hint**: Concatenate the state using LangGraph's reducer pattern (`Annotated` + `operator.add`), rather than replacing the state value.

In [ ]:
# Install dependencies if needed:
# !pip install langgraph langchain-core

## Method 1: Concatenating State Directly on `name`

By annotating `name` with `operator.add` as the reducer in `AgentState`, any node returning a update to `name` will **concatenate** with the existing value instead of overwriting it.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# 1. Define the State with a concatenation reducer
class AgentState(TypedDict):
    name: Annotated[str, operator.add]

# 2. Define the node that appends the compliment message to the state
def compliment_node(state: AgentState):
    return {"name": ", you're doing an amazing job learning LangGraph!"}

# 3. Build the graph
workflow = StateGraph(AgentState)
workflow.add_node("compliment_node", compliment_node)
workflow.add_edge(START, "compliment_node")
workflow.add_edge("compliment_node", END)

# 4. Compile the graph
app = workflow.compile()

# 5. Invoke with Input: {"name": "Bob"}
input_data = {"name": "Bob"}
result = app.invoke(input_data)

# Display Output
output = result["name"]
print("Final State:", result)
print("Output:", output)

## Method 2: Retaining `name` and Concatenating into an `output` field

If you want to keep the original input `name` unmodified and concatenate multiple segments into an `output` state key across nodes:

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

class ComplimentState(TypedDict):
    name: str
    output: Annotated[str, operator.add]

def start_compliment(state: ComplimentState):
    # Append the name to output
    return {"output": state["name"]}

def finish_compliment(state: ComplimentState):
    # Append the compliment to output
    return {"output": ", you're doing an amazing job learning LangGraph!"}

builder = StateGraph(ComplimentState)
builder.add_node("start_compliment", start_compliment)
builder.add_node("finish_compliment", finish_compliment)

builder.add_edge(START, "start_compliment")
builder.add_edge("start_compliment", "finish_compliment")
builder.add_edge("finish_compliment", END)

agent = builder.compile()
result = agent.invoke({"name": "Bob"})

print("Final State:", result)
print("Output:", result["output"])